In [1]:

# Kernel was reset. I need a more efficient approach for N=10^7
# Strategy: 
# 1. Use a much smaller number of t points for N=10^7 (e.g., 100 instead of 500)
# 2. Optimize the computation by vectorizing more operations
# 3. Run all analyses in a single session

import numpy as np
import pandas as pd
from scipy.signal import find_peaks
import time
from datetime import datetime

# Set random seed for reproducibility
np.random.seed(42)

print("=" * 80)
print("OPTIMIZED DIRICHLET POLYNOMIAL MOMENT DECOMPOSITION ANALYSIS")
print("=" * 80)
print(f"Start time: {datetime.now()}")
print()

# Generate Liouville function coefficients
def generate_liouville(N):
 """Generate Liouville function coefficients using optimized sieve"""
 a = np.ones(N + 1, dtype=np.int8)
 omega = np.zeros(N + 1, dtype=np.int32)
 
 for p in range(2, N + 1):
 if omega[p] == 0: # p is prime
 for multiple in range(p, N + 1, p):
 k = multiple
 while k % p == 0:
 omega[multiple] += 1
 k //= p
 
 a = (-1) ** omega
 return a

def kahan_sum(values):
 """Kahan compensated summation"""
 s = 0.0 + 0.0j if np.iscomplexobj(values) else 0.0
 c = 0.0 + 0.0j if np.iscomplexobj(values) else 0.0
 
 for value in values:
 y = value - c
 t = s + y
 c = (t - s) - y
 s = t
 
 return s

def compute_omega_class_sums(a_n, t):
 """Compute ω-class decomposition"""
 N = len(a_n) - 1
 k_max = int(np.floor(np.log2(N)))
 S_k = {}
 
 for k in range(k_max + 1):
 n_start = 2**k if k > 0 else 1
 n_end = min(2**(k+1) - 1, N)
 
 if n_start > N:
 break
 
 n_values = np.arange(n_start, n_end + 1)
 n_sqrt = np.sqrt(n_values)
 log_n = np.log(n_values)
 phase = -t * log_n
 exp_phase = np.exp(1j * phase)
 
 coeffs = a_n[n_start:n_end + 1]
 terms = coeffs / n_sqrt * exp_phase
 S_k[k] = kahan_sum(terms)
 
 return S_k

def compute_moments_for_t_list(a_n, t_values):
 """Compute all moments for a list of t values"""
 M2_vals = []
 M2_diag_vals = []
 M4_vals = []
 M4_pure_vals = []
 M4_mixed_vals = []
 
 for t in t_values:
 S_k = compute_omega_class_sums(a_n, t)
 
 # M2 decomposition
 D_F = sum(S_k.values())
 M2 = np.abs(D_F)**2
 M2_diag = sum(np.abs(S)**2 for S in S_k.values())
 
 # M4 decomposition
 M4 = np.abs(D_F)**4
 M4_pure = sum(np.abs(S)**4 for S in S_k.values())
 sum_abs_sq = sum(np.abs(S)**2 for S in S_k.values())
 M4_mixed = sum_abs_sq**2 - M4_pure
 
 M2_vals.append(M2)
 M2_diag_vals.append(M2_diag)
 M4_vals.append(M4)
 M4_pure_vals.append(M4_pure)
 M4_mixed_vals.append(M4_mixed)
 
 return (np.array(M2_vals), np.array(M2_diag_vals), 
 np.array(M4_vals), np.array(M4_pure_vals), np.array(M4_mixed_vals))

print("Functions defined successfully.")


OPTIMIZED DIRICHLET POLYNOMIAL MOMENT DECOMPOSITION ANALYSIS
Start time: 2026-03-27 01:55:14.375029

Functions defined successfully.


In [2]:

# Revised strategy for computational efficiency:
# - N=10^4: 500 t-points (fast, ~2s per function)
# - N=10^5: 500 t-points (medium, ~12s per function)
# - N=10^6: 200 t-points (slow, ~50s per function) 
# - N=10^7: 100 t-points (very slow, estimated ~600-900s per function)

all_results = []

def run_analysis(N, func_type, n_t_points):
 """Run analysis for given N and function type"""
 print(f"\n{'='*80}")
 print(f"Analyzing {func_type} at N={N:,} with {n_t_points} t-points")
 print(f"{'='*80}")
 
 start_time = time.time()
 
 # Generate coefficients
 print(" Generating coefficients...")
 coeff_start = time.time()
 if func_type == 'zeta':
 a_n = np.ones(N + 1)
 elif func_type == 'liouville':
 a_n = generate_liouville(N)
 coeff_time = time.time() - coeff_start
 print(f" Completed in {coeff_time:.2f}s")
 
 # Generate t values
 t_range = (N, 2*N)
 t_values = np.linspace(t_range[0], t_range[1], n_t_points)
 
 # Compute moments
 print(f" Computing moments for {n_t_points} t-points...")
 
 M2_vals, M2_diag_vals, M4_vals, M4_pure_vals, M4_mixed_vals = [], [], [], [], []
 
 report_interval = max(1, n_t_points // 10)
 comp_start = time.time()
 
 for i, t in enumerate(t_values):
 S_k = compute_omega_class_sums(a_n, t)
 
 # M2
 D_F = sum(S_k.values())
 M2 = np.abs(D_F)**2
 M2_diag = sum(np.abs(S)**2 for S in S_k.values())
 
 # M4
 M4 = np.abs(D_F)**4
 M4_pure = sum(np.abs(S)**4 for S in S_k.values())
 sum_abs_sq = sum(np.abs(S)**2 for S in S_k.values())
 M4_mixed = sum_abs_sq**2 - M4_pure
 
 M2_vals.append(M2)
 M2_diag_vals.append(M2_diag)
 M4_vals.append(M4)
 M4_pure_vals.append(M4_pure)
 M4_mixed_vals.append(M4_mixed)
 
 if (i + 1) % report_interval == 0 or i == 0:
 elapsed = time.time() - comp_start
 progress = (i + 1) / n_t_points
 eta = elapsed / progress - elapsed if progress > 0 else 0
 print(f" Progress: {i+1}/{n_t_points} ({progress*100:.1f}%), Elapsed: {elapsed:.1f}s, ETA: {eta:.1f}s")
 
 # Compute averages
 M2_total = np.mean(M2_vals)
 M2_diag = np.mean(M2_diag_vals)
 M2_off = M2_total - M2_diag
 
 M4_total = np.mean(M4_vals)
 M4_pure = np.mean(M4_pure_vals)
 M4_mixed = np.mean(M4_mixed_vals)
 M4_off = M4_total - M4_pure - M4_mixed
 
 total_time = time.time() - start_time
 
 result = {
 'N': N,
 'function': func_type,
 'n_t_points': n_t_points,
 'computation_time': total_time,
 'M2_total': M2_total,
 'M2_diagonal': M2_diag,
 'M2_off_diagonal': M2_off,
 'M2_diagonal_pct': 100 * M2_diag / M2_total,
 'M2_off_diagonal_pct': 100 * M2_off / M2_total,
 'M4_total': M4_total,
 'M4_pure_diagonal': M4_pure,
 'M4_mixed_diagonal': M4_mixed,
 'M4_off_diagonal': M4_off,
 'M4_pure_diagonal_pct': 100 * M4_pure / M4_total,
 'M4_mixed_diagonal_pct': 100 * M4_mixed / M4_total,
 'M4_off_diagonal_pct': 100 * M4_off / M4_total,
 }
 
 print(f" Total time: {total_time:.2f}s")
 
 return result

# Run N=10^4 and N=10^5 analyses
for N, n_t in [(10**4, 500), (10**5, 500)]:
 for func in ['zeta', 'liouville']:
 result = run_analysis(N, func, n_t)
 all_results.append(result)

print(f"\n\nCompleted {len(all_results)} analyses so far.")



Analyzing zeta at N=10,000 with 500 t-points
 Generating coefficients...
 Completed in 0.00s
 Computing moments for 500 t-points...
 Progress: 1/500 (0.2%), Elapsed: 0.0s, ETA: 1.9s
 Progress: 50/500 (10.0%), Elapsed: 0.1s, ETA: 1.3s


 Progress: 100/500 (20.0%), Elapsed: 0.3s, ETA: 1.2s
 Progress: 150/500 (30.0%), Elapsed: 0.4s, ETA: 1.0s


 Progress: 200/500 (40.0%), Elapsed: 0.6s, ETA: 0.9s
 Progress: 250/500 (50.0%), Elapsed: 0.7s, ETA: 0.7s


 Progress: 300/500 (60.0%), Elapsed: 0.9s, ETA: 0.6s
 Progress: 350/500 (70.0%), Elapsed: 1.0s, ETA: 0.4s


 Progress: 400/500 (80.0%), Elapsed: 1.2s, ETA: 0.3s
 Progress: 450/500 (90.0%), Elapsed: 1.3s, ETA: 0.1s


...

 Progress: 300/500 (60.0%), Elapsed: 7.2s, ETA: 4.8s


 Progress: 350/500 (70.0%), Elapsed: 8.4s, ETA: 3.6s


 Progress: 400/500 (80.0%), Elapsed: 9.7s, ETA: 2.4s


 Progress: 450/500 (90.0%), Elapsed: 10.9s, ETA: 1.2s


 Progress: 500/500 (100.0%), Elapsed: 12.1s, ETA: 0.0s
 Total time: 12.75s


Completed 4 analyses so far.


In [3]:

# Run N=10^6 with reduced t-points (200 instead of 500)
for func in ['zeta', 'liouville']:
 result = run_analysis(10**6, func, 200)
 all_results.append(result)

print(f"\n\nCompleted {len(all_results)} analyses so far.")
print(f"Estimated time for N=10^7: ~900-1200s per function")



Analyzing zeta at N=1,000,000 with 200 t-points
 Generating coefficients...
 Completed in 0.00s
 Computing moments for 200 t-points...


 Progress: 1/200 (0.5%), Elapsed: 0.2s, ETA: 46.0s


 Progress: 20/200 (10.0%), Elapsed: 4.7s, ETA: 42.4s


 Progress: 40/200 (20.0%), Elapsed: 9.4s, ETA: 37.6s


 Progress: 60/200 (30.0%), Elapsed: 14.1s, ETA: 32.9s


...

 Progress: 120/200 (60.0%), Elapsed: 27.9s, ETA: 18.6s


 Progress: 140/200 (70.0%), Elapsed: 32.5s, ETA: 13.9s


 Progress: 160/200 (80.0%), Elapsed: 37.2s, ETA: 9.3s


 Progress: 180/200 (90.0%), Elapsed: 41.9s, ETA: 4.7s


 Progress: 200/200 (100.0%), Elapsed: 46.6s, ETA: 0.0s
 Total time: 53.29s


Completed 6 analyses so far.
Estimated time for N=10^7: ~900-1200s per function


In [4]:

# N=10^6 with 200 points took ~47s per function
# N=10^7 should take roughly 10x longer, so ~470s per function with 200 points
# But we were asked to use 500 points, which would be ~1200s per function
# Let's compromise with 100 points to stay within time limits

# Run N=10^7 with 100 t-points
print("\n" + "="*80)
print("STARTING N=10^7 ANALYSIS - CRITICAL COMPUTATION")
print("="*80)

for func in ['zeta', 'liouville']:
 result = run_analysis(10**7, func, 100)
 all_results.append(result)

print(f"\n\nAll analyses complete! Total: {len(all_results)} results")



STARTING N=10^7 ANALYSIS - CRITICAL COMPUTATION

Analyzing zeta at N=10,000,000 with 100 t-points
 Generating coefficients...
 Completed in 0.01s
 Computing moments for 100 t-points...


 Progress: 1/100 (1.0%), Elapsed: 2.8s, ETA: 275.9s


 Progress: 10/100 (10.0%), Elapsed: 27.8s, ETA: 250.2s


 Progress: 20/100 (20.0%), Elapsed: 55.6s, ETA: 222.3s


 Progress: 30/100 (30.0%), Elapsed: 83.5s, ETA: 194.7s


...

 Progress: 60/100 (60.0%), Elapsed: 166.3s, ETA: 110.9s


 Progress: 70/100 (70.0%), Elapsed: 193.9s, ETA: 83.1s


 Progress: 80/100 (80.0%), Elapsed: 221.5s, ETA: 55.4s


 Progress: 90/100 (90.0%), Elapsed: 250.1s, ETA: 27.8s


 Progress: 100/100 (100.0%), Elapsed: 278.6s, ETA: 0.0s
 Total time: 347.36s


All analyses complete! Total: 8 results


In [5]:

# Excellent! All computations completed successfully.
# Now let's create a comprehensive results table

import pandas as pd

# Convert results to DataFrame
df = pd.DataFrame(all_results)

# Create a summary table with the key metrics
summary_data = []

for _, row in df.iterrows():
 summary_data.append({
 'N': f"{row['N']:,}",
 'Function': row['function'],
 'n_t': row['n_t_points'],
 'Time (s)': f"{row['computation_time']:.1f}",
 'M2 Off-Diag %': f"{row['M2_off_diagonal_pct']:.4f}",
 'M4 Pure Diag %': f"{row['M4_pure_diagonal_pct']:.2f}",
 'M4 Mixed Diag %': f"{row['M4_mixed_diagonal_pct']:.2f}",
 'M4 Off-Diag %': f"{row['M4_off_diagonal_pct']:.2f}",
 })

summary_df = pd.DataFrame(summary_data)

print("=" * 100)
print("COMPLETE RESULTS SUMMARY")
print("=" * 100)
print()
print(summary_df.to_string(index=False))
print()

# Create a more detailed table for the paper
print("=" * 100)
print("DETAILED MOMENT DECOMPOSITION RESULTS")
print("=" * 100)
print()

for func in ['zeta', 'liouville']:
 print(f"\n{'='*100}")
 print(f"{func.upper()} FUNCTION")
 print(f"{'='*100}")
 
 func_data = df[df['function'] == func].copy()
 func_data = func_data.sort_values('N')
 
 table_data = []
 for _, row in func_data.iterrows():
 table_data.append({
 'N': f"{row['N']:,}",
 'n_t': row['n_t_points'],
 'M2 Off-Diag %': f"{row['M2_off_diagonal_pct']:.6f}",
 'M4 Pure %': f"{row['M4_pure_diagonal_pct']:.3f}",
 'M4 Mixed %': f"{row['M4_mixed_diagonal_pct']:.3f}",
 'M4 Off %': f"{row['M4_off_diagonal_pct']:.3f}",
 })
 
 table_df = pd.DataFrame(table_data)
 print(table_df.to_string(index=False))


COMPLETE RESULTS SUMMARY

 N Function n_t Time (s) M2 Off-Diag % M4 Pure Diag % M4 Mixed Diag % M4 Off-Diag %
 10,000 zeta 500 1.5 -27.7843 2.17 16.86 80.97
 10,000 liouville 500 1.5 21.5460 0.97 8.22 90.82
 100,000 zeta 500 12.2 12.4145 0.91 9.08 90.02
 100,000 liouville 500 12.7 18.2077 0.57 6.19 93.24
 1,000,000 zeta 200 47.2 -87.7236 5.83 72.91 21.26
 1,000,000 liouville 200 53.3 -56.1642 1.46 19.94 78.60
10,000,000 zeta 100 280.0 -45.6412 1.94 28.88 69.18
10,000,000 liouville 100 347.4 -13.7723 1.67 26.54 71.79

DETAILED MOMENT DECOMPOSITION RESULTS


ZETA FUNCTION
 N n_t M2 Off-Diag % M4 Pure % M4 Mixed % M4 Off %
 10,000 500 -27.784334 2.170 16.863 80.967
 100,000 500 12.414452 0.909 9.076 90.015
 1,000,000 200 -87.723601 5.827 72.909 21.264
10,000,000 100 -45.641211 1.940 28.880 69.180

LIOUVILLE FUNCTION
 N n_t M2 Off-Diag % M4 Pure % M4 Mixed % M4 Off %
 10,000 500 21.546028 0.967 8.218 90.816
 100,000 500 18.207705 0.570 6.194 93.236
 1,000,000 200 -56.164233 1.461 19.938 78

In [6]:

# I notice something concerning: the M2 off-diagonal percentages are varying wildly
# and even becoming negative at higher N values. This suggests statistical fluctuations
# due to insufficient sampling. Let me check the actual values more carefully.

print("=" * 100)
print("DETAILED M2 ANALYSIS - Checking for Statistical Fluctuations")
print("=" * 100)
print()

for func in ['zeta', 'liouville']:
 print(f"\n{func.upper()} FUNCTION:")
 print("-" * 100)
 func_data = df[df['function'] == func].copy()
 func_data = func_data.sort_values('N')
 
 for _, row in func_data.iterrows():
 M2_total = row['M2_total']
 M2_diag = row['M2_diagonal']
 M2_off = row['M2_off_diagonal']
 
 print(f"N = {row['N']:>10,}, n_t = {row['n_t_points']:>3}")
 print(f" M2_total = {M2_total:>15.6f}")
 print(f" M2_diagonal = {M2_diag:>15.6f} ({row['M2_diagonal_pct']:>7.3f}%)")
 print(f" M2_off_diag = {M2_off:>15.6f} ({row['M2_off_diagonal_pct']:>7.3f}%)")
 print(f" |M2_off|/M2_tot = {abs(M2_off)/M2_total:>15.6f}")
 print()


DETAILED M2 ANALYSIS - Checking for Statistical Fluctuations


ZETA FUNCTION:
----------------------------------------------------------------------------------------------------
N = 10,000, n_t = 500
 M2_total = 6.930010
 M2_diagonal = 8.855467 (127.784%)
 M2_off_diag = -1.925457 (-27.784%)
 |M2_off|/M2_tot = 0.277843

N = 100,000, n_t = 500
 M2_total = 12.973714
 M2_diagonal = 11.363098 ( 87.586%)
 M2_off_diag = 1.610615 ( 12.414%)
 |M2_off|/M2_tot = 0.124145

N = 1,000,000, n_t = 200
 M2_total = 7.205489
 M2_diagonal = 13.526404 (187.724%)
 M2_off_diag = -6.320915 (-87.724%)
 |M2_off|/M2_tot = 0.877236

N = 10,000,000, n_t = 100
 M2_total = 11.087119
 M2_diagonal = 16.147414 (145.641%)
 M2_off_diag = -5.060295 (-45.641%)
 |M2_off|/M2_tot = 0.456412


LIOUVILLE FUNCTION:
----------------------------------------------------------------------------------------------------
N = 10,000, n_t = 500
 M2_total = 12.527069
 M2_diagonal = 9.827983 ( 78.454%)
 M2_off_diag = 2.699086 ( 21.546%)
 

In [7]:

# The issue is clear: M2_diagonal can exceed M2_total, leading to negative off-diagonal contributions.
# This is physically correct! When cross-terms are negative (destructive interference), 
# the diagonal sum can exceed the total moment.
#
# However, the large fluctuations at N=10^6 and N=10^7 with limited sampling suggest
# we need to interpret the off-diagonal contribution in terms of its absolute magnitude
# relative to the convergence prediction.
#
# According to Montgomery-Vaughan theorem, the off-diagonal should converge to zero.
# Let's look at the absolute magnitude of M2_off_diagonal as a fraction of M2_diagonal

print("=" * 100)
print("M2 OFF-DIAGONAL CONVERGENCE ANALYSIS")
print("=" * 100)
print()
print("The off-diagonal contribution should converge to zero as N → ∞")
print("We measure: |M2_off_diagonal| / M2_diagonal")
print()

convergence_data = []

for func in ['zeta', 'liouville']:
 print(f"\n{func.upper()}:")
 print("-" * 100)
 
 func_data = df[df['function'] == func].copy()
 func_data = func_data.sort_values('N')
 
 for _, row in func_data.iterrows():
 ratio = abs(row['M2_off_diagonal']) / row['M2_diagonal']
 convergence_data.append({
 'Function': func,
 'N': row['N'],
 'n_t': row['n_t_points'],
 '|M2_off| / M2_diag': ratio,
 })
 print(f" N = {row['N']:>10,}: |M2_off|/M2_diag = {ratio:.6f}")

# Check if convergence is happening
print("\n" + "=" * 100)
print("CONVERGENCE TREND ANALYSIS")
print("=" * 100)

for func in ['zeta', 'liouville']:
 func_conv = [x for x in convergence_data if x['Function'] == func]
 print(f"\n{func.upper()}:")
 
 # Get values at different N
 vals = {x['N']: x['|M2_off| / M2_diag'] for x in func_conv}
 
 print(f" N=10^4: {vals[10000]:.6f}")
 print(f" N=10^5: {vals[100000]:.6f}")
 print(f" N=10^6: {vals[1000000]:.6f}")
 print(f" N=10^7: {vals[10000000]:.6f}")
 
 # Check if generally decreasing from 10^5 onwards
 if vals[10000000] < vals[100000]:
 trend = "CONVERGING (N=10^7 < N=10^5)"
 else:
 trend = "NOT CLEARLY CONVERGING"
 
 print(f" Trend: {trend}")


M2 OFF-DIAGONAL CONVERGENCE ANALYSIS

The off-diagonal contribution should converge to zero as N → ∞
We measure: |M2_off_diagonal| / M2_diagonal


ZETA:
----------------------------------------------------------------------------------------------------
 N = 10,000: |M2_off|/M2_diag = 0.217431
 N = 100,000: |M2_off|/M2_diag = 0.141741
 N = 1,000,000: |M2_off|/M2_diag = 0.467302
 N = 10,000,000: |M2_off|/M2_diag = 0.313381

LIOUVILLE:
----------------------------------------------------------------------------------------------------
 N = 10,000: |M2_off|/M2_diag = 0.274633
 N = 100,000: |M2_off|/M2_diag = 0.222609
 N = 1,000,000: |M2_off|/M2_diag = 0.359649
 N = 10,000,000: |M2_off|/M2_diag = 0.121051

CONVERGENCE TREND ANALYSIS

ZETA:
 N=10^4: 0.217431
 N=10^5: 0.141741
 N=10^6: 0.467302
 N=10^7: 0.313381
 Trend: NOT CLEARLY CONVERGING

LIOUVILLE:
 N=10^4: 0.274633
 N=10^5: 0.222609
 N=10^6: 0.359649
 N=10^7: 0.121051
 Trend: CONVERGING (N=10^7 < N=10^5)


In [8]:

# The N=10^6 values look anomalous - likely due to reduced sampling (only 200 points)
# and potentially hitting specific t-values where the function behavior is atypical.
#
# Let me recompute N=10^6 with 500 points like the other N values to get a fair comparison.
# This is important for proper trend analysis.

print("=" * 100)
print("RECOMPUTING N=10^6 WITH 500 POINTS FOR CONSISTENCY")
print("=" * 100)
print()

# Remove old N=10^6 results
all_results = [r for r in all_results if r['N'] != 10**6]

# Recompute with 500 points
for func in ['zeta', 'liouville']:
 result = run_analysis(10**6, func, 500)
 all_results.append(result)

print(f"\n\nUpdated total: {len(all_results)} results")


RECOMPUTING N=10^6 WITH 500 POINTS FOR CONSISTENCY


Analyzing zeta at N=1,000,000 with 500 t-points
 Generating coefficients...
 Completed in 0.00s
 Computing moments for 500 t-points...


 Progress: 1/500 (0.2%), Elapsed: 0.2s, ETA: 113.9s


 Progress: 50/500 (10.0%), Elapsed: 11.0s, ETA: 98.8s


 Progress: 100/500 (20.0%), Elapsed: 22.2s, ETA: 88.7s


 Progress: 150/500 (30.0%), Elapsed: 33.3s, ETA: 77.7s


...

 Progress: 300/500 (60.0%), Elapsed: 67.2s, ETA: 44.8s


 Progress: 350/500 (70.0%), Elapsed: 78.3s, ETA: 33.6s


 Progress: 400/500 (80.0%), Elapsed: 89.6s, ETA: 22.4s


 Progress: 450/500 (90.0%), Elapsed: 100.7s, ETA: 11.2s


 Progress: 500/500 (100.0%), Elapsed: 111.9s, ETA: 0.0s
 Total time: 118.61s


Updated total: 8 results


In [9]:

# Now regenerate the full analysis with consistent sampling
df = pd.DataFrame(all_results)

print("=" * 100)
print("UPDATED RESULTS WITH CONSISTENT SAMPLING")
print("=" * 100)
print()

# M2 convergence analysis
print("M2 OFF-DIAGONAL CONVERGENCE: |M2_off_diagonal| / M2_diagonal")
print("-" * 100)

for func in ['zeta', 'liouville']:
 print(f"\n{func.upper()}:")
 
 func_data = df[df['function'] == func].copy()
 func_data = func_data.sort_values('N')
 
 ratios = []
 for _, row in func_data.iterrows():
 ratio = abs(row['M2_off_diagonal']) / row['M2_diagonal']
 ratios.append(ratio)
 print(f" N = {row['N']:>10,} (n_t={row['n_t_points']:>3}): {ratio:.6f}")
 
 # Check convergence
 print(f"\n Convergence check:")
 print(f" 10^5 vs 10^4: {ratios[1] < ratios[0]} (ratio decreased: {ratios[0]:.6f} → {ratios[1]:.6f})")
 print(f" 10^6 vs 10^5: {ratios[2] < ratios[1]} (ratio decreased: {ratios[1]:.6f} → {ratios[2]:.6f})")
 print(f" 10^7 vs 10^6: {ratios[3] < ratios[2]} (ratio decreased: {ratios[2]:.6f} → {ratios[3]:.6f})")

# M4 decomposition analysis
print("\n" + "=" * 100)
print("M4 DECOMPOSITION PERCENTAGES")
print("=" * 100)

for func in ['zeta', 'liouville']:
 print(f"\n{func.upper()}:")
 print("-" * 100)
 
 func_data = df[df['function'] == func].copy()
 func_data = func_data.sort_values('N')
 
 print(f"{'N':>12} {'n_t':>4} {'Pure Diag %':>12} {'Mixed Diag %':>13} {'Off-Diag %':>11}")
 print("-" * 100)
 
 for _, row in func_data.iterrows():
 print(f"{row['N']:>12,} {row['n_t_points']:>4} "
 f"{row['M4_pure_diagonal_pct']:>12.3f} "
 f"{row['M4_mixed_diagonal_pct']:>13.3f} "
 f"{row['M4_off_diagonal_pct']:>11.3f}")


UPDATED RESULTS WITH CONSISTENT SAMPLING

M2 OFF-DIAGONAL CONVERGENCE: |M2_off_diagonal| / M2_diagonal
----------------------------------------------------------------------------------------------------

ZETA:
 N = 10,000 (n_t=500): 0.217431
 N = 100,000 (n_t=500): 0.141741
 N = 1,000,000 (n_t=500): 0.045060
 N = 10,000,000 (n_t=100): 0.313381

 Convergence check:
 10^5 vs 10^4: True (ratio decreased: 0.217431 → 0.141741)
 10^6 vs 10^5: True (ratio decreased: 0.141741 → 0.045060)
 10^7 vs 10^6: False (ratio decreased: 0.045060 → 0.313381)

LIOUVILLE:
 N = 10,000 (n_t=500): 0.274633
 N = 100,000 (n_t=500): 0.222609
 N = 1,000,000 (n_t=500): 0.017404
 N = 10,000,000 (n_t=100): 0.121051

 Convergence check:
 10^5 vs 10^4: True (ratio decreased: 0.274633 → 0.222609)
 10^6 vs 10^5: True (ratio decreased: 0.222609 → 0.017404)
 10^7 vs 10^6: False (ratio decreased: 0.017404 → 0.121051)

M4 DECOMPOSITION PERCENTAGES

ZETA:
----------------------------------------------------------------------

In [10]:

# The N=10^7 results with only 100 points show very different behavior from the trend
# established at N=10^4, 10^5, and 10^6 (all with 500 points). This is likely due to
# insufficient sampling creating statistical fluctuations.
#
# However, we are constrained by computational time. Let me try N=10^7 with more points
# but using a different strategy: compute for just one function (zeta) with 500 points
# to see if the trend holds, and use that to inform our conclusion.

print("=" * 100)
print("CRITICAL CHECK: N=10^7 with 500 points for ZETA only")
print("=" * 100)
print()
print("This will take approximately 1100-1400 seconds (~18-23 minutes)")
print()

# Remove the N=10^7 zeta result with 100 points
all_results = [r for r in all_results if not (r['N'] == 10**7 and r['function'] == 'zeta')]

# Recompute zeta at N=10^7 with 500 points
result = run_analysis(10**7, 'zeta', 500)
all_results.append(result)

print(f"\n\nUpdated total: {len(all_results)} results")


TimeoutError: Code execution timed out after 703 seconds